In [1]:
import findspark
import numpy as np
import pandas as pd
import subprocess
import joblib
import time

from tqdm.auto import tqdm

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
findspark.init()

In [3]:
spark = SparkSession.builder \
    .appName("US Accidents Analysis - Optimized for MSI GF63") \
    .master("local[10]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "12") \
    .config("spark.default.parallelism", "12") \
    .config("spark.network.timeout", "600s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.sql.broadcastTimeout", "300s") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .config("spark.python.worker.reuse", "true") \
    .getOrCreate()

spark

In [4]:
print(
    subprocess.check_output(
        "nvidia-smi",
        shell=True
    ).decode()
)

Sat May 16 09:04:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 582.53                 Driver Version: 582.53         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Quadro M1000M                WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A    0C    P8            N/A  /  200W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
DATA_PATH = "Data/Feature_Engineered_Data"

In [6]:
train_df = spark.read.parquet(f"{DATA_PATH}/train")
test_df  = spark.read.parquet(f"{DATA_PATH}/test")

In [7]:
print("Train:", train_df.count())
print("Test:", test_df.count())

Train: 6987064
Test: 1747953


In [8]:
test_df.printSchema()

root
 |-- Severity: integer (nullable = true)
 |-- Start_Time: string (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- Visibility(mi): double (nullable = true)
 |-- Wind_Direction: string (nullable = true)
 |-- Wind_Speed(mph): double (nullable = true)
 |-- Precipitation(in): double (nullable = true)
 |-- Weather_Condition: string (nullable = true)
 |-- Amenity: integer (nullable = true)
 |-- Bump: integer (nullable = true)
 |-- Crossing: integer (nullable = true)
 |-- Give_Way: integer (nullable = true)
 |-- Junction: integer (nullable = true)
 |-- No_Exit: integer 

In [9]:
train_df = train_df.withColumn(
    "label",
    col("Severity")
)

test_df = test_df.withColumn(
    "label",
    col("Severity")
)

In [10]:
train_df = train_df.select(
    "features",
    "label",
    "weight"
)

test_df = test_df.select(
    "features",
    "label",
    "weight"
)

train_df.cache()
test_df.cache()

print(
    "Train Samples:",
    train_df.count()
)

print(
    "Test Samples:",
    test_df.count()
)

Train Samples: 6987064
Test Samples: 1747953


In [11]:
def spark_to_numpy_stream(df):

    total_rows = df.count()

    X = []
    y = []
    w = []

    for row in tqdm(
            df.toLocalIterator(),
            total=total_rows,
            desc="Converting",
            unit="rows"
    ):

        X.append(
            row["features"].toArray()
        )

        y.append(
            row["label"]
        )

        w.append(
            row["weight"]
        )

    return (
        np.array(X),
        np.array(y),
        np.array(w)
    )

In [12]:
X_train, y_train, w_train = (
    spark_to_numpy_stream(
        train_df
    )
)
print(
    "Train Data Shapes:",
    X_train.shape,
    y_train.shape,
    w_train.shape
)

Converting:   0%|          | 0/6987064 [00:00<?, ?rows/s]

Train Data Shapes: (6987064, 39) (6987064,) (6987064,)


In [13]:
X_test, y_test, w_test = (
    spark_to_numpy_stream(
        test_df
    )
)
print(
    "Test Data Shapes:",
    X_test.shape,
    y_test.shape,
    w_test.shape
)

Converting:   0%|          | 0/1747953 [00:00<?, ?rows/s]

Test Data Shapes: (1747953, 39) (1747953,) (1747953,)


In [31]:
# CatBoost parameter mapping from LightGBM:
# n_estimators      -> iterations
# objective         -> loss_function  ('multiclass' -> 'MultiClass')
# num_class         -> inferred automatically from labels
# max_depth         -> depth          (CatBoost depth is symmetric; 8 is equivalent)
# num_leaves        -> removed        (CatBoost uses symmetric trees; not applicable)
# subsample         -> subsample      (same name; requires bootstrap_type='Bernoulli')
# colsample_bytree  -> rsm            (Random Subspace Method, same concept)
# random_state      -> random_seed
# device='gpu'      -> task_type='GPU'
# gpu_device_id     -> devices        (string '0' for first GPU)

model = CatBoostClassifier(

    loss_function='MultiClass',

    iterations=300,

    learning_rate=0.05,

    depth=8,

    bootstrap_type='Bernoulli',
    subsample=0.8,

    # rsm=0.8  ← remove this

    random_seed=42,

    task_type='GPU',
    devices='0',

    verbose=100
)

In [32]:
start = time.time()

print(
    "Training Started..."
)

model.fit(

    X_train,

    y_train,

    sample_weight=w_train,

    eval_set=(
        X_test,
        y_test
    )
)

end = time.time()

print(
    f"Training Time: {(end - start) / 60:.2f} minutes"
)

Training Started...
0:	learn: 1.3375042	test: 1.3598950	best: 1.3598950 (0)	total: 1.45s	remaining: 7m 14s
100:	learn: 0.7688433	test: 1.0071310	best: 1.0071310 (100)	total: 2m 29s	remaining: 4m 55s
200:	learn: 0.7249906	test: 0.9632283	best: 0.9632283 (200)	total: 4m 50s	remaining: 2m 22s
299:	learn: 0.7024171	test: 0.9395120	best: 0.9395120 (299)	total: 7m 8s	remaining: 0us
bestTest = 0.9395119606
bestIteration = 299
Training Time: 7.31 minutes


In [33]:
start = time.time()

print(
    "Predicting..."
)

y_pred = model.predict(
    X_test
).flatten()  # CatBoost returns a 2-D column vector; flatten to 1-D

end = time.time()

print(
    f"Prediction Time: {end - start:.2f} sec"
)

Predicting...
Prediction Time: 4.14 sec


In [34]:
acc = accuracy_score(
    y_test,
    y_pred
)

print(
    "Accuracy:",
    acc
)

Accuracy: 0.5252017645783382


In [35]:
f1 = f1_score(
    y_test,
    y_pred,
    average='weighted'
)

print(
    "Weighted F1:",
    f1
)

Weighted F1: 0.6053600177087187


In [36]:
print(

    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           1       0.08      0.91      0.14     14604
           2       0.96      0.48      0.64   1412570
           3       0.41      0.67      0.51    270299
           4       0.10      0.82      0.17     50480

    accuracy                           0.53   1747953
   macro avg       0.39      0.72      0.37   1747953
weighted avg       0.84      0.53      0.61   1747953



In [37]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[ 13221    307    707    369]
 [126889 683111 252328 350242]
 [ 28811  20235 180367  40886]
 [   985   6392   1774  41329]]


In [38]:
# Option 1 – save with joblib (same as the original LightGBM notebook)
joblib.dump(

    model,

    "Models/catboost_model.pkl"
)

# Option 2 – save with CatBoost's native format (recommended; preserves all metadata)
model.save_model("Models/catboost_model.cbm")

print(
    "Model Saved"
)

Model Saved
